In [ ]:
import io, base64, numpy as np, torch, sys
from PIL import Image
from IPython.display import display, Javascript, HTML

# --- CROSS-CORRELATION ALGORITHM ---
def fft_cross_correlation(imgA_b64, imgB_b64, wsize, overlap):
    try:
        a_data = base64.b64decode(imgA_b64.split(',')[1])
        b_data = base64.b64decode(imgB_b64.split(',')[1])
        img1_pil = Image.open(io.BytesIO(a_data))
        img2_pil = Image.open(io.BytesIO(b_data))

        #Store Original Resolution
        orig_w, orig_h = img1_pil.size

        img1_gray = np.array(img1_pil.convert("L"), dtype=np.float32)
        img2_gray = np.array(img2_pil.convert("L"), dtype=np.float32)

        wy, wx = int(wsize), int(wsize)
        stride_y, stride_x = max(wy-int(overlap), 1), max(wx-int(overlap), 1)
        fft_size = 2 ** int(np.ceil(np.log2(wy)))
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        t1, t2 = torch.from_numpy(img1_gray).to(device), torch.from_numpy(img2_gray).to(device)
        ny, nx = (img1_gray.shape[0]-wy)//stride_y + 1, (img1_gray.shape[1]-wx)//stride_x + 1
        p1 = torch.nn.functional.unfold(t1[None,None,...], (wy, wx), stride=(stride_y, stride_x)).squeeze(0).T
        p2 = torch.nn.functional.unfold(t2[None,None,...], (wy, wx), stride=(stride_y, stride_x)).squeeze(0).T
        p1 -= p1.mean(dim=1, keepdim=True); p2 -= p2.mean(dim=1, keepdim=True)
        denom = (torch.norm(p1, dim=1) * torch.norm(p2, dim=1)).clamp_min(1e-8)
        F1 = torch.fft.rfft2(p1.view(-1, wy, wx), s=(fft_size, fft_size))
        F2 = torch.fft.rfft2(p2.view(-1, wy, wx), s=(fft_size, fft_size))
        corr = torch.fft.irfft2(F1 * torch.conj(F2), s=(fft_size, fft_size))
        corr_max = (torch.amax(corr, dim=(1,2)) / denom).view(ny, nx).cpu().numpy()

        #Generate Result Array
        res_arr = (255 - ((corr_max - corr_max.min()) / (corr_max.max() - corr_max.min() + 1e-8) * 255)).astype(np.uint8)

        res_img = Image.fromarray(res_arr).resize((orig_w, orig_h), resample=Image.NEAREST)

        buf = io.BytesIO()
        res_img.save(buf, format="PNG")
        return base64.b64encode(buf.getvalue()).decode('utf-8')
    except Exception as e: return f"Error: {str(e)}"

if 'google.colab' in sys.modules:
    from google.colab import output
    output.register_callback('notebook.run_fft', fft_cross_correlation)

# --- UI ---
UI_HTML = """
<!DOCTYPE html><html><head>
<script src="https://cdn.tailwindcss.com"></script>
<style>
 body { background-color: #FCFBF4; color: #f8fafc; margin: 0; padding: 40px; }
 .card { background: #FFFFED; border: 1px solid #1e293b; border-radius: 1.5rem; }
 </style>
</head><body>
    <div class="max-w-6xl mx-auto">
        <header class="flex justify-between items-center mb-10 border-b border-slate-800 pb-6">
            <h1 class="text-3xl font-black text-amber-800 ">Cross-Correlation Interface</span></h1>
        </header>
        <div class="grid grid-cols-1 lg:grid-cols-4 gap-8">
            <aside class="card p-6 h-fit space-y-6">
                <h2 class="text-xs font-bold text-slate-800 uppercase tracking-widest">Parameters</h2>
                <div class="space-y-4">
                    <div>
                        <div class="flex justify-between text-[10px] text-slate-500 font-bold uppercase mb-1">
                            <span>Window Size</span><span id="ws_val" class="text-slate-600">16</span>
                        </div>
                        <input type="range" id="ws" min="8" max="128" step="8" value="16" class="w-full accent-amber-800">
                    </div>
                    <div>
                        <div class="flex justify-between text-[10px] text-slate-500 font-bold uppercase mb-1">
                            <span>Overlap</span><span id="ov_val" class="text-slate-600">8</span>
                        </div>
                        <input type="range" id="ov" min="0" max="64" step="4" value="8" class="w-full accent-amber-800">
                    </div>
                </div>
                <div class="border-t border-slate-800 pt-6 space-y-4">
                    <input type="file" id="fA" class="text-[10px] w-full text-slate-600 file:bg-amber-800 file:border-0 file:rounded-full file:px-2 file:py-1 file:text-white">
                    <input type="file" id="fB" class="text-[10px] w-full text-slate-600 file:bg-amber-800 file:border-0 file:rounded-full file:px-2 file:py-1 file:text-white">
                </div>
                <button id="run" class="w-full bg-amber-800 py-4 rounded-xl font-black hover:bg-amber-700 transition-all uppercase text-sm">Execute</button>
            </aside>

            <section class="lg:col-span-3 grid grid-cols-2 gap-6">
                <div class="card p-4 min-h-[380px] flex flex-col">
                    <span class="text-[9px] font-mono text-slate-800 mb-2 uppercase tracking-widest">Frame 1</span>
                    <img id="vA" class="max-h-80 object-contain m-auto">
                </div>
                <div class="card p-4 min-h-[380px] flex flex-col">
                    <span class="text-[9px] font-mono text-slate-800 mb-2 uppercase tracking-widest">Frame 2</span>
                    <img id="vB" class="max-h-80 object-contain m-auto">
                </div>
            </section>
        </div>
        <div id="resContainer" class="mt-10 card p-1 bg-[#FFFFED] border-slate-800">
            <div id="resPlaceholder" class="placeholder-dashed rounded-[1.4rem] py-24 flex flex-col items-center justify-center">
                <div id="placeIcon" class="text-slate-500/20 mb-4 transition-all duration-300">
                    <svg class="w-16 h-16" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="1" d="M4 16l4.586-4.586a2 2 0 012.828 0L16 16m-2-2l1.586-1.586a2 2 0 012.828 0L20 14m-6-6h.01M6 20h12a2 2 0 002-2V6a2 2 0 00-2-2H6a2 2 0 00-2 2v12a2 2 0 002 2z"></path></svg>
                </div>
                <p id="placeText" class="text-slate-800 font-bold uppercase text-[10px] tracking-[0.3em]">Waiting for Execution</p>
            </div>

            <div id="resDisplay" class="hidden bg-slate-950 rounded-[1.4rem] p-12 text-center">
                <h3 class="text-xs font-bold text-slate-800 uppercase tracking-[0.5em] mb-8">Cross-Correlation Result</h3>
                <img id="vRes" class="mx-auto rounded-lg shadow-2xl border border-slate-500/20">
            </div>
        </div>

        <!--
        <div id="resBox" class="hidden mt-8 p-8 bg-slate-900 rounded-3xl border border-slate-500/20 text-center">
            <img id="vRes" class="mx-auto rounded-lg shadow-2xl max-w-full">
        </div>
        -->
    </div>
    <script>
        const ws = document.getElementById('ws'); const ov = document.getElementById('ov');
        ws.oninput = () => document.getElementById('ws_val').innerText = ws.value;
        ov.oninput = () => document.getElementById('ov_val').innerText = ov.value;

        const hF = (id, v) => document.getElementById(id).onchange = (e) => {
            const r = new FileReader(); r.onload = (ev) => document.getElementById(v).src = ev.target.result;
            r.readAsDataURL(e.target.files[0]);
        };
        hF('fA', 'vA'); hF('fB', 'vB');

        document.getElementById('run').onclick = () => {
            const btn = document.getElementById('run');
            btn.innerText = "CALCULATING..."; btn.disabled = true;
            window.opener.postMessage({
                type: 'RUN_FFT',
                imgA: document.getElementById('vA').src,
                imgB: document.getElementById('vB').src,
                ws: ws.value,
                ov: ov.value
            }, '*');
        };

        window.addEventListener('message', (e) => {
            if(e.data.type === 'FFT_DONE') {
                document.getElementById('vRes').src = 'data:image/png;base64,' + e.data.data;
                document.getElementById('resPlaceholder').classList.add('hidden');
                document.getElementById('resDisplay').classList.remove('hidden');
                document.getElementById('run').innerText = "EXECUTE";
                document.getElementById('run').disabled = false;
            }
        });
    </script>
</body></html>
"""

import uuid
import time
from google.colab import output

# --- KERNEL CONTROLLER ---
btn_id = f"btn_{uuid.uuid4().hex[:8]}"

display(HTML(f"""
    <div style="padding:20px; background:#FCFBF4; border-radius:12px; border:1px solid #1e293b; color:#94a3b8; font-family:sans-serif;">
        <button id="{btn_id}" style="background:#92400E; color:white; border:none; padding:10px 20px; border-radius:6px; cursor:pointer; font-weight:bold;">
            Workspace Ready
        </button>
        <p id="msg_{btn_id}" style="color:black margin-top:10px; font-size:11px;">Click Manually to open worskspace if popup window is blocked.</p>
    </div>
"""))

js_code = r'''
window.launchFFTWorkspace = function() {
    let childWin = window.open("", "FFT_WINDOW", "width=window.innerwidth,height=window.innerheight");
    if (childWin) {
        childWin.document.open();
        childWin.document.write(`''' + UI_HTML + r'''`);
        childWin.document.close();

        window.addEventListener('message', async (event) => {
            if (event.data.type === 'RUN_FFT') {
                try {
                    const result = await google.colab.kernel.invokeFunction('notebook.run_fft',
                        [event.data.imgA, event.data.imgB, event.data.ws, event.data.ov], {});
                    const b64 = result.data['text/plain'].replace(/^'|'$/g, "");
                    childWin.postMessage({type: 'FFT_DONE', data: b64}, '*');
                } catch(e) { console.error(e); }
            }
        }, { once: false });
    }
};

document.getElementById("''' + btn_id + r'''").onclick = () => window.launchFFTWorkspace();
'''

display(Javascript(js_code))

# TRIGGER: This only runs when the "Play" button is hit
time.sleep(0.5)
output.eval_js('window.launchFFTWorkspace()')

<IPython.core.display.Javascript object>